# Rasterizer

Oftentimes we want to make 2d images plotting something versus something else, and we'd like to sift through a huge amount of data to do it. 

In [ ]:

from typing import Callable, Optional
import numpy as np
import matplotlib.pyplot as plot
from scipy.sparse import coo_matrix  # renamed coo_array in later versions of scipy


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

class Raster2DConfig:
    """
    Defines a 2-D raster grid over arbitrary x and y axes.

    Parameters
    ----------
    x_min, x_max   : bounds of the x axis
    y_min, y_max   : bounds of the y axis
    resolution      : bin width (same units as x/y, applied to both axes)
    x_resolution    : optional per-axis override for x bin width
    y_resolution    : optional per-axis override for y bin width
    x_label         : human-readable name for the x axis (e.g. "colour")
    y_label         : human-readable name for the y axis (e.g. "abs_magnitude")
    """

    def __init__(
        self,
        x_min: float,
        x_max: float,
        y_min: float,
        y_max: float,
        resolution: float = 0.02,
        x_resolution: Optional[float] = None,
        y_resolution: Optional[float] = None,
        x_label: str = "x",
        y_label: str = "y",
    ):
        self.x_min = x_min
        self.x_max = x_max
        self.y_min = y_min
        self.y_max = y_max
        self.x_label = x_label
        self.y_label = y_label

        self.x_resolution = x_resolution if x_resolution is not None else resolution
        self.y_resolution = y_resolution if y_resolution is not None else resolution

        # derived — computed once
        self.x_range  = x_max - x_min
        self.y_range  = y_max - y_min
        self.xmax_idx = int(self.x_range / self.x_resolution)
        self.ymax_idx = int(self.y_range / self.y_resolution)

    def __repr__(self):
        return (
            f"Raster2DConfig("
            f"{self.x_label}=[{self.x_min}, {self.x_max}], "
            f"{self.y_label}=[{self.y_min}, {self.y_max}], "
            f"x_res={self.x_resolution}, y_res={self.y_resolution}, "
            f"grid={self.xmax_idx}x{self.ymax_idx})"
        )



def raster_index(x: float, y: float, cfg: Raster2DConfig) -> int:
    """
    Compute a unique 1-D raster index for a point (x, y) on the configured grid.

    Returns 0 for points outside the grid boundary (sentinel value).

    Parameters
    ----------
    x   : value along the x axis
    y   : value along the y axis
    cfg : Raster2DConfig instance
    """
    xidx = int(round((x - cfg.x_min) * cfg.xmax_idx / cfg.x_range))
    yidx = int(round((y - cfg.y_min) * cfg.ymax_idx / cfg.y_range))

    if xidx < 0 or xidx >= cfg.xmax_idx or yidx < 0 or yidx >= cfg.ymax_idx:
        return 0

    return xidx + (cfg.xmax_idx * yidx)


# ---------------------------------------------------------------------------
# Derived-quantity transforms
# Callers supply a `transform` callable that maps raw columns -> (x, y).
# This keeps the raster logic clean and lets the CMD distance-modulus step
# (or any other physics) live in one place at the call site.
# ---------------------------------------------------------------------------

def raster_index_transformed(
    transform: Callable[..., tuple],
    cfg: Raster2DConfig,
    *args,
) -> int:
    """
    Apply `transform(*args) -> (x, y)` then compute the raster index.

    """
    x, y = transform(*args)
    return raster_index(x, y, cfg)


# ---------------------------------------------------------------------------
# Vectorised inverse transforms 
# ---------------------------------------------------------------------------

def xidx_from_raster(ridx_array: np.ndarray, cfg: Raster2DConfig) -> np.ndarray:
    """Recover x bin indices from a vector of raster indices."""
    return np.mod(ridx_array, cfg.xmax_idx)


def yidx_from_raster(ridx_array: np.ndarray, cfg: Raster2DConfig) -> np.ndarray:
    """Recover y bin indices from a vector of raster indices."""
    return np.trunc(ridx_array / cfg.xmax_idx)


def x_from_raster(ridx_array: np.ndarray, cfg: Raster2DConfig) -> np.ndarray:
    """Recover x values (bin centres) from a vector of raster indices."""
    return cfg.x_min + (xidx_from_raster(ridx_array, cfg) * cfg.x_range / cfg.xmax_idx)


def y_from_raster(ridx_array: np.ndarray, cfg: Raster2DConfig) -> np.ndarray:
    """Recover y values (bin centres) from a vector of raster indices."""
    return cfg.y_min + (yidx_from_raster(ridx_array, cfg) * cfg.y_range / cfg.ymax_idx)


# ---------------------------------------------------------------------------
# Engine-specific registration helper
# ---------------------------------------------------------------------------
def register_pyspark_udf(
    spark,
    cfg: Raster2DConfig,
    transform: Optional[Callable] = None,
    udf_name: str = "rasterize",
):
    """
    Register raster_index as a Spark SQL / DataFrame UDF.

    If `transform` is provided, the UDF accepts the raw columns that transform
    expects and calls raster_index_transformed internally. Otherwise the UDF
    expects (x, y) directly.

    Parameters
    ----------
    spark     : active SparkSession
    cfg       : Raster2DConfig instance
    transform : optional callable(*raw_cols) -> (x, y)
    udf_name  : name to register under for SQL API

    Returns
    -------
    The PySpark UDF object (for use with the DataFrame API).

    Usage — direct (x, y):
        register_pyspark_udf(spark, cfg)
        spark.sql("SELECT rasterize(colour, abs_mag) FROM sources")

    Usage — with transform (e.g. CMD distance modulus):
        register_pyspark_udf(spark, cfg, transform=cmd_transform)
        spark.sql("SELECT rasterize(magnitude, colour, parallax) FROM sources")
    """
    from pyspark.sql.functions import udf
    from pyspark.sql.types import IntegerType

    if transform is not None:
        def _udf(*args):
            return raster_index_transformed(transform, cfg, *args)
    else:
        def _udf(x, y):
            return raster_index(x, y, cfg)

    spark.udf.register(udf_name, _udf, IntegerType())
    return udf(_udf, IntegerType())


# ---------------------------------------------------------------------------
# Plotting
# ---------------------------------------------------------------------------

def plot_raster(
    pdf,
    cfg,
    title="Raster diagram",
    cmap=None,
    figsize=(8.0, 12.0),
    log_counts=True,
):
    """
    Plot a 2D rasterized count grid from a pandas/Spark-collected DataFrame
    containing 'ridx' and 'count_in_pixel' columns.

    Parameters
    ----------
    pdf         : pandas DataFrame with columns 'ridx', 'count_in_pixel'
    cfg         : Raster2DConfig instance used to produce the raster indices
    title       : plot title
    cmap        : optional matplotlib colormap name
    figsize     : figure size tuple
    log_counts  : whether to log-scale counts (enhances low-density features)
    """
    # recover 2D bin coordinates from the 1D raster index
    xidx = xidx_from_raster(pdf["ridx"].values, cfg)
    yidx = yidx_from_raster(pdf["ridx"].values, cfg)

    counts = pdf["count_in_pixel"].values
    values = np.log(counts) if log_counts else counts

    sparse_data = coo_matrix(
        (values, (yidx, xidx)),
        shape=(cfg.ymax_idx, cfg.xmax_idx),
    )
    dense_data = sparse_data.todense()

    plot.figure(figsize=figsize)
    plot.title(title, fontsize=16)
    plot.xlabel(cfg.x_label, fontsize=14)
    plot.ylabel(cfg.y_label, fontsize=14)
    plot.imshow(
        dense_data,
        aspect="auto",
        cmap=cmap,
        origin='lower',
        extent=[
            cfg.x_min,
            cfg.x_min + cfg.xmax_idx * cfg.x_resolution,
            cfg.y_min,
            cfg.y_min + cfg.ymax_idx * cfg.y_resolution,
        ],
    )
    plot.colorbar(label="log(count)" if log_counts else "count")
    plot.show()

The above code defines a 2D rasterization framework in Python, allowing users to create a grid over specified x and y axes, compute unique raster indices for points, and visualize the resulting rasterized data. It includes configuration options, transformation functions, and plotting capabilities.

The next bit is a specific set of options to make a CMD.

In [2]:

def cmd_config(
    brightest_abs_mag: float = -5.0,
    faintest_abs_mag: float  = 15.0,
    bluest_colour: float     = -2.0,
    reddest_colour: float    = +6.0,
    mag_resolution: float    = 0.02,
) -> Raster2DConfig:
    """Colour-magnitude diagram — x=colour, y=absolute magnitude."""
    return Raster2DConfig(
        x_min=bluest_colour,
        x_max=reddest_colour,
        y_min=brightest_abs_mag,
        y_max=faintest_abs_mag,
        resolution=mag_resolution,
        x_label="colour",
        y_label="abs_magnitude",
    )



This next bit is specific to the AstroFlow setup, but will provide an example of how to make a CMD.

In [ ]:
# Import the spark config
from astroflow_spark_gaia import spark

In [ ]:
cfg = cmd_config()
register_pyspark_udf(spark, cfg, udf_name="rasterize")  # expects (x, y) directly

query = (
    "SELECT rasterize("
    "  g.phot_g_mean_mag - ag_gspphot - t.k_m, "                                   # x = colour
    "  (g.phot_g_mean_mag - ag_gspphot) - (5.0 * LOG10(1000.0 / g.parallax) - 5.0) " # y = abs mag
    ") AS ridx, COUNT(*) AS count_in_pixel "
    "FROM gaiadr3.gaia_source AS g "
    "INNER JOIN gaiaedr3.gaia_source_tmasspsc_best_neighbours AS t "
    "  ON g.source_id = t.source_id "
    "WHERE g.ruwe < 1.4 AND g.parallax_over_error > 10.0 "
    "  AND t.k_m IS NOT NULL AND g.ag_gspphot IS NOT NULL "
    "GROUP BY ridx HAVING ridx >= 0"
)

df = spark.sql(query)

In [ ]:
cfg.x_label = r"(G - K)$_0$ / mag"
cfg.y_label = r"M$_G$ - A$_G$ / mag"

plot_raster(pdf, cfg, title="Dereddened optical/IR CAMD for the Gaia DR3 catalogue")